# Notebook 02 - Analisis Fiscal Argentina 2020-2026

**Fuente:** Secretaria de Hacienda | AIF (Sector Publico Base Caja) + IMIG  
**Deflactor:** IPC Nivel General INDEC  
**Unidad:** pesos constantes de abril 2026 (salvo indicacion contraria)

**Alcance titulares:** Sector Publico Total = Administracion Nacional + PAMI + Fondos Fiduciarios  
**Alcance desglose:** Administracion Nacional (AIF) y IMIG funcional  
**Unico mes sin datos AIF mensual:** Jun-2022 (solo existe acumulado I Semestre)

In [ ]:
# ── Celda 1: Dependencias y funciones globales ────────────────────────
import sys, io, os, zipfile, requests
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install -q pandas matplotlib openpyxl

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import matplotlib.dates as mdates
import numpy as np
import warnings
warnings.filterwarnings('ignore')

DPI      = 600
FIG_W    = 26
FIG_H    = 7
BAR_W    = 0.8   # para integer positions: ancho relativo
GRAFICOS = []
GAP_DATE = pd.Timestamp('2022-06-01')  # unico mes sin datos AIF mensual

plt.rcParams['figure.dpi']        = 100
plt.rcParams['savefig.dpi']       = DPI
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['font.size']         = 9

MESES_ES = {1:'Ene',2:'Feb',3:'Mar',4:'Abr',5:'May',6:'Jun',
            7:'Jul',8:'Ago',9:'Sep',10:'Oct',11:'Nov',12:'Dic'}

def drop_gap(serie):
    """Elimina Jun-2022 para cerrar el gap visual en los graficos."""
    return serie.drop(GAP_DATE, errors='ignore')

def fmt_int(ax, idx, interval=3):
    """
    Etiquetas en español cada N meses sobre eje con posiciones enteras.
    Los meses de inicio de trimestre (Ene, Abr, Jul, Oct) se etiquetan.
    """
    meses_tick = [1, 4, 7, 10]
    ticks, labels = [], []
    for i, d in enumerate(idx):
        if d.month in meses_tick:
            ticks.append(i)
            labels.append(f"{MESES_ES[d.month]}-{d.year}")
    ax.set_xticks(ticks)
    ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=7)
    ax.set_xlim(-0.8, len(idx) - 0.2)

def save_fig(fig, nombre):
    fname = f'{nombre}.png'
    fig.savefig(fname, dpi=DPI, bbox_inches='tight')
    GRAFICOS.append(fname)
    print(f'  Guardado: {fname}')

print('OK')

In [ ]:
# ── Celda 2: Carga de datos ───────────────────────────────────────────
REPO     = 'https://raw.githubusercontent.com/santiagoriverti/cuentas_publicas/main'
REPO_BIN = 'https://github.com/santiagoriverti/cuentas_publicas/raw/main'

df_aif  = pd.read_csv(f'{REPO}/output/aif_consolidado.csv',  parse_dates=['fecha'])
df_imig_raw = pd.read_csv(f'{REPO}/output/imig_consolidado.csv', parse_dates=['fecha'])
# Deduplicar IMIG: un valor por (fecha, concepto, nivel)
df_imig = df_imig_raw.drop_duplicates(subset=['fecha','concepto_codigo','nivel_jerarquia']).copy()
df      = df_aif[df_aif['periodo'] == 'mensual'].copy()

ipc_raw = pd.read_excel(
    io.BytesIO(requests.get(f'{REPO_BIN}/data/reference/IPC.xlsx').content),
    usecols=['date', 'Nivel general'])
ipc_raw['date'] = pd.to_datetime(ipc_raw['date'])
ipc = ipc_raw.set_index('date')['Nivel general'].sort_index()
ipc.index = ipc.index.to_period('M').to_timestamp()

BASE     = ipc.index.max()
IPC_BASE = ipc.loc[BASE]

def deflactar(serie):
    idx = serie.index.to_period('M').to_timestamp()
    return serie.values * (IPC_BASE / ipc.reindex(idx).values)

def get_serie(concepto, subsector='total_adm_nacional'):
    mask = (df['concepto_codigo'] == concepto) & (df['subsector'] == subsector)
    return df[mask].set_index('fecha')['valor_millones_pesos'].sort_index()

def get_serie_total(concepto):
    s_nac  = get_serie(concepto, 'total_adm_nacional')
    s_pami = get_serie(concepto, 'pami_fdos_otros')
    s_gen  = get_serie(concepto, 'total_general')
    s_suma = s_nac.add(s_pami.reindex(s_nac.index).fillna(0))
    if len(s_gen) > 0:
        s_suma.update(s_gen)
    return s_suma.sort_index()

def get_imig_anual(concepto, nivel=None):
    """Suma anual en pesos constantes de un concepto IMIG."""
    mask = df_imig['concepto_codigo'] == concepto
    if nivel is not None:
        mask &= df_imig['nivel_jerarquia'] == nivel
    s = df_imig[mask].copy()
    s['ipc_v'] = ipc.reindex(s['fecha'].dt.to_period('M').dt.to_timestamp()).values
    s['real']  = s['valor_millones_pesos'] * (IPC_BASE / s['ipc_v'])
    return s.groupby(s['fecha'].dt.year)['real'].sum() / 1e6

print(f'AIF mensual  : {len(df):,} registros | {df.fecha.min().strftime("%Y-%m")} - {df.fecha.max().strftime("%Y-%m")}')
print(f'IMIG (dedup) : {len(df_imig):,} registros')
print(f'IPC          : {ipc.index.min().strftime("%Y-%m")} - {ipc.index.max().strftime("%Y-%m")}')
print(f'Base deflac. : {BASE.strftime("%Y-%m")} (IPC = {IPC_BASE:,.0f})')
print(f'Gap removido : {GAP_DATE.strftime("%b-%Y")} (solo tiene acumulado I Semestre)')

In [ ]:
# ── Celda 3: Resultado Primario y Financiero ──────────────────────────
primario   = drop_gap(get_serie_total('XIV_RESULTADO_PRIMARIO'))
financiero = drop_gap(get_serie_total('XV_RESULTADO_FINANCIERO'))
prim_real  = deflactar(primario)
fin_real   = deflactar(financiero)
idx        = primario.index
pos        = list(range(len(idx)))

fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))
colors = ['#27ae60' if v >= 0 else '#c0392b' for v in prim_real]
ax.bar(pos, prim_real / 1e6, color=colors, alpha=0.9, label='Resultado Primario', zorder=2)
ax.plot([p + 0.5 for p in pos], fin_real / 1e6, 'o-', color='#2c3e50',
        lw=1.8, ms=4, label='Resultado Financiero', zorder=3)
ax.axhline(0, color='black', lw=1.0)
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:,.1f}'))
ax.set_ylabel(f'Billones ARS ({BASE.strftime("%b %Y")})')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.25, zorder=0)
fmt_int(ax, idx)
plt.tight_layout()
save_fig(fig, '01_resultado_primario_financiero')
plt.show()

df_anual = pd.DataFrame({'Primario nominal (B)': primario/1e6,
                          'Financiero nominal (B)': financiero/1e6,
                          'Primario real (B)': prim_real/1e6,
                          'Financiero real (B)': fin_real/1e6})
print(f'Resumen anual - Sector Publico Total (billones ARS, real = precios {BASE.strftime("%b %Y")}):')
print(df_anual.groupby(df_anual.index.year).sum().round(1).to_string())

In [ ]:
# ── Celda 4: Ajuste fiscal - titulares y desglose ─────────────────────
print('Sector Publico Total (real, billones ARS):')
for concepto, label in [
    ('XIV_RESULTADO_PRIMARIO',               'Resultado Primario'),
    ('XV_RESULTADO_FINANCIERO',              'Resultado Financiero'),
    ('XII_GASTOS_PRIMARIOS_DESPUES_FIGURAT', 'Gasto Primario'),
]:
    s = get_serie_total(concepto)
    s_real = pd.Series(deflactar(s), index=s.index)
    anual  = s_real.groupby(s_real.index.year).sum() / 1e6
    v24 = anual.get(2024,0) - anual.get(2023,0)
    v25 = anual.get(2025,0) - anual.get(2023,0)
    print(f'  {label}: 2022={anual.get(2022,0):.1f}  2023={anual.get(2023,0):.1f}  '
          f'2024={anual.get(2024,0):.1f}  2025={anual.get(2025,0):.1f}  '
          f'| Var 23->24: {v24:+.1f}  Var 23->25: {v25:+.1f}')

print()
print('Desglose Administracion Nacional (real, billones ARS):')
conceptos_ajuste = {
    'I_INGRESOS_CORRIENTES':        'Ingresos corrientes',
    'II_GASTOS_CORRIENTES':         'Gastos corrientes',
    'II2_INTERESES':                '  Intereses',
    'II3_PRESTACIONES_SEG_SOCIAL':  '  Prestaciones Seg.Social',
    'II1a_REMUNERACIONES':          '  Remuneraciones',
    'II4b1_TRANSF_PROVINCIAS_CABA': '  Transf. Provincias/CABA',
    'II4b2_TRANSF_UNIVERSIDADES':   '  Universidades',
    'II4a_TRANSF_SECTOR_PRIVADO':   '  Transf. Privado (subsidios)',
    'V_GASTOS_CAPITAL':             'Gastos de capital',
    'XIV_RESULTADO_PRIMARIO':       'RESULTADO PRIMARIO',
}
rows = []
for cod, nombre in conceptos_ajuste.items():
    s = get_serie(cod)
    s_real = pd.Series(deflactar(s), index=s.index)
    for yr in [2022, 2023, 2024, 2025]:
        rows.append({'Concepto': nombre, 'Anio': yr,
                     'Real': s_real[s_real.index.year == yr].sum()})

df_ajuste = pd.DataFrame(rows).pivot(index='Concepto', columns='Anio', values='Real') / 1e6
df_ajuste['Var 23->24'] = df_ajuste[2024] - df_ajuste[2023]
df_ajuste['Var 23->25'] = df_ajuste[2025] - df_ajuste[2023]
df_ajuste['% ajuste 23->25'] = (
    df_ajuste['Var 23->25'] /
    abs(df_ajuste.loc['Gastos corrientes','Var 23->25'] +
        df_ajuste.loc['Gastos de capital','Var 23->25']) * 100
).round(1)
df_ajuste = df_ajuste.reindex(list(conceptos_ajuste.values()))
print(df_ajuste.round(1).to_string())

# Grafico barras horizontales
mask_det = df_ajuste.index.str.startswith('  ')
vals = df_ajuste.loc[mask_det, 'Var 23->25'].dropna().sort_values()
etiquetas = vals.index.str.strip()
fig, ax = plt.subplots(figsize=(13, 6))
colors  = ['#1a5276' if v >= 0 else '#922b21' for v in vals]
bars = ax.barh(etiquetas, vals.values, color=colors, alpha=0.92,
               edgecolor='white', linewidth=0.5, height=0.65)
ax.axvline(0, color='#2c3e50', lw=1.2)
for bar, val, concepto in zip(bars, vals.values, etiquetas):
    pct = df_ajuste.loc[f'  {concepto}', '% ajuste 23->25']
    offset = 0.3 if val >= 0 else -0.3
    ax.text(val + offset, bar.get_y() + bar.get_height()/2,
            f'{val:+.1f} B  ({abs(pct):.1f}%)', va='center',
            ha='left' if val >= 0 else 'right', fontsize=8.5, fontweight='bold')
ax.set_xlabel(f'Variacion real 2023->2025 | Billones ARS ({BASE.strftime("%b %Y")})')
ax.set_xlim(vals.min() - 3, max(vals.max() + 5, 2))
ax.grid(axis='x', alpha=0.2)
ax.spines['left'].set_visible(False)
ax.text(0.99, 0.02, 'Rojo = recorte  |  Azul = aumento',
        transform=ax.transAxes, ha='right', va='bottom', fontsize=8, color='gray', style='italic')
plt.tight_layout()
save_fig(fig, '02_ajuste_componentes_2023_2025')
plt.show()

In [ ]:
# ── Celda 5: Transferencias a provincias ──────────────────────────────
corr_tes = drop_gap(get_serie('II4b1_TRANSF_PROVINCIAS_CABA', 'tesoro_nacional'))
cap_tes  = drop_gap(get_serie('V2a_TRANSF_CAPITAL_PROVINCIAS', 'tesoro_nacional'))
corr_tot = drop_gap(get_serie('II4b1_TRANSF_PROVINCIAS_CABA', 'total_adm_nacional'))
cap_tot  = drop_gap(get_serie('V2a_TRANSF_CAPITAL_PROVINCIAS', 'total_adm_nacional'))

total_prov      = corr_tot.add(cap_tot.reindex(corr_tot.index).fillna(0))
corr_real       = pd.Series(deflactar(corr_tes), index=corr_tes.index)
cap_real        = pd.Series(deflactar(cap_tes),  index=cap_tes.index)
total_prov_real = pd.Series(deflactar(total_prov), index=total_prov.index)
idx = corr_real.index
pos = list(range(len(idx)))

fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))
corr_vals = corr_real.values / 1000
cap_vals  = cap_real.reindex(idx).fillna(0).values / 1000
ax.bar(pos, corr_vals, color='#6c3483', alpha=0.9, label='Corrientes (Tesoro)', zorder=2)
ax.bar(pos, cap_vals, bottom=corr_vals, color='#ca6f1e', alpha=0.9,
       label='Capital (Tesoro)', zorder=2)
ax.set_ylabel(f'Miles de MM ARS ({BASE.strftime("%b %Y")})')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.25, zorder=0)
fmt_int(ax, idx)
plt.tight_layout()
save_fig(fig, '03_transferencias_provincias')
plt.show()

gasto_tot  = get_serie_total('XII_GASTOS_PRIMARIOS_DESPUES_FIGURAT')
gasto_real = pd.Series(deflactar(gasto_tot), index=gasto_tot.index)
resumen    = pd.DataFrame({
    'Transf. prov. (B)':     total_prov_real.groupby(total_prov_real.index.year).sum() / 1e6,
    'Gasto prim. total (B)': gasto_real.groupby(gasto_real.index.year).sum() / 1e6,
})
resumen['Prov./Gasto (%)'] = resumen['Transf. prov. (B)'] / resumen['Gasto prim. total (B)'] * 100
var_prov    = resumen.loc[2024,'Transf. prov. (B)'] - resumen.loc[2023,'Transf. prov. (B)']
var_prov25  = resumen.loc[2025,'Transf. prov. (B)'] - resumen.loc[2023,'Transf. prov. (B)']
var_gasto   = resumen.loc[2024,'Gasto prim. total (B)'] - resumen.loc[2023,'Gasto prim. total (B)']
var_gasto25 = resumen.loc[2025,'Gasto prim. total (B)'] - resumen.loc[2023,'Gasto prim. total (B)']
print(resumen.round(1).to_string())
print(f'\nAjuste 2023->2024: {var_prov:+.1f} B ({var_prov/resumen.loc[2023,"Transf. prov. (B)"]*100:.0f}%) | {var_prov/var_gasto*100:.1f}% del ajuste')
print(f'Ajuste 2023->2025: {var_prov25:+.1f} B ({var_prov25/resumen.loc[2023,"Transf. prov. (B)"]*100:.0f}%) | {var_prov25/var_gasto25*100:.1f}% del ajuste')

In [ ]:
# ── Celda 6: Composicion del gasto corriente ──────────────────────────
comp_gasto = {
    'II3_PRESTACIONES_SEG_SOCIAL': 'Prestaciones Seg.Social',
    'II2_INTERESES':               'Intereses',
    'II1a_REMUNERACIONES':         'Remuneraciones',
    'II4a_TRANSF_SECTOR_PRIVADO':  'Subsidios',
    'II4b1_TRANSF_PROVINCIAS_CABA':'Transf. Provincias',
    'II4b2_TRANSF_UNIVERSIDADES':  'Universidades',
}
colors_comp = ['#c0392b','#8e44ad','#2980b9','#16a085','#6c3483','#d68910']

idx_full = get_serie('II_GASTOS_CORRIENTES').index
idx  = idx_full[idx_full != GAP_DATE]
pos  = list(range(len(idx)))
fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))
bottom = np.zeros(len(idx))

for (cod, label), color in zip(comp_gasto.items(), colors_comp):
    vals = deflactar(get_serie(cod).reindex(idx).fillna(0)) / 1e6
    ax.bar(pos, vals, bottom=bottom, label=label, color=color, alpha=0.88, zorder=2)
    bottom += vals

ax.set_ylabel(f'Billones ARS ({BASE.strftime("%b %Y")})')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:,.1f}'))
ax.legend(loc='upper left', fontsize=8, ncol=2)
ax.grid(axis='y', alpha=0.25, zorder=0)
fmt_int(ax, idx)
plt.tight_layout()
save_fig(fig, '04_composicion_gasto_corriente')
plt.show()

In [ ]:
# ── Celda 8: Torta - distribucion del recorte de gasto 2023->2025 ─────\n# Fuente: IMIG (funcional). Rubros con recortes entre 2023 y 2025.\n\nrubros_imig = {\n    'Gastos_capital':              'Obra publica',\n    'Subsidios_economicos':        'Subsidios',\n    'Transf_corrientes_provincias':'Transf. Provincias',\n    'Salarios':                    'Salarios',\n    'Jubilaciones_pensiones':      'Jubilaciones',\n    'Transf_universidades':        'Universidades',\n    'Pensiones_no_contributivas':  'Pensiones NC',\n    'INSSJP_PAMI':                 'PAMI',\n    'Otros_prog_sociales':         'Otros prog. sociales',\n    'AUH':                         'AUH',\n}\n\nvariaciones = {}\nfor cod, nombre in rubros_imig.items():\n    anual = get_imig_anual(cod)\n    if 2023 in anual.index and 2025 in anual.index:\n        var = anual[2025] - anual[2023]\n        if nombre not in variaciones or abs(var) > abs(variaciones[nombre]):\n            variaciones[nombre] = var\n\nrecortes  = {k: -v for k, v in variaciones.items() if v < 0}\naumentos  = {k:  v for k, v in variaciones.items() if v > 0}\ntotal_rec = sum(recortes.values())\n\nprint(f'Reduccion gasto (IMIG, real abr-2026):')\nprint(f'  Total recortes identificados: {total_rec:.1f} B ARS')\nprint(f'  Rubros con recorte: {list(recortes.keys())}')\nprint(f'  Rubros con aumento: {list(aumentos.keys())}')\nprint()\n\n# Grafico de torta\nlabels     = list(recortes.keys())\nsizes      = list(recortes.values())\ncolors_pie = ['#922b21','#1a5276','#6c3483','#1e8449','#784212',\n              '#0e6655','#4d5656','#2e4053','#7d6608']\n\nfig, ax = plt.subplots(figsize=(10, 8))\nwedges, texts, autotexts = ax.pie(\n    sizes, labels=labels, autopct='%1.1f%%',\n    colors=colors_pie[:len(sizes)], startangle=140,\n    pctdistance=0.75, labeldistance=1.12,\n    wedgeprops={'edgecolor': 'white', 'linewidth': 1.2}\n)\nfor at in autotexts:\n    at.set_fontsize(8)\n    at.set_fontweight('bold')\n    at.set_color('white')\nfor t in texts:\n    t.set_fontsize(9)\n\nax.set_title(f'Distribucion del recorte de gasto 2023->2025\\n'\n             f'Total identificado en IMIG: {total_rec:.1f} B ARS (precios {BASE.strftime(\"%b %Y\")})',\n             fontsize=11, pad=15)\nplt.tight_layout()\nsave_fig(fig, '06_torta_recorte_gasto')\nplt.show()

In [ ]:
# ── Celda 9: Analisis completo del ajuste fiscal Milei ────────────────
# Fuentes: AIF (Sector Publico Total) para macro | IMIG para funcional

# PIB nominal aproximado (en billones = 10^12 pesos)
PIB_B = {2020: 44.9, 2021: 72.0, 2022: 115.0,
         2023: 143.2, 2024: 586.7, 2025: 725.0}
# 2023: derivado de deficit 8.74B = -6.1% PIB (Hacienda)
# 2024: derivado de superavit 1.76B = +0.3% PIB (Hacienda)
# 2025: derivado de superavit 1.45B = +0.2% PIB (Hacienda)

# ── 1. Resultado fiscal anual ─────────────────────────────────────────
print('=' * 80)
print('1. RESULTADO FISCAL ANUAL - SECTOR PUBLICO TOTAL')
print('   (Billones ARS, real = precios abril 2026)')
print('=' * 80)

df_macro = pd.DataFrame()
for concepto, col in [
    ('XIV_RESULTADO_PRIMARIO',               'Primario_real'),
    ('XV_RESULTADO_FINANCIERO',              'Financiero_real'),
    ('I_INGRESOS_CORRIENTES',                'Ingresos_real'),
    ('XII_GASTOS_PRIMARIOS_DESPUES_FIGURAT', 'Gasto_primario_real'),
    ('II2_INTERESES',                        'Intereses_real'),
]:
    s = get_serie_total(concepto)
    real = pd.Series(deflactar(s), index=s.index)
    df_macro[col] = real.groupby(real.index.year).sum() / 1e6

for concepto, col in [
    ('XIV_RESULTADO_PRIMARIO',  'Primario_nom'),
    ('XV_RESULTADO_FINANCIERO', 'Financiero_nom'),
]:
    s = get_serie_total(concepto)
    df_macro[col] = s.groupby(s.index.year).sum() / 1e6

df_macro['Primario_pct_PIB']   = df_macro.apply(
    lambda r: f"{r['Primario_nom']/PIB_B.get(r.name,1)*100:+.1f}%"
    if r.name in PIB_B else 'n/d', axis=1)
df_macro['Financiero_pct_PIB'] = df_macro.apply(
    lambda r: f"{r['Financiero_nom']/PIB_B.get(r.name,1)*100:+.1f}%"
    if r.name in PIB_B else 'n/d', axis=1)

tabla1 = df_macro[['Primario_real','Financiero_real','Primario_pct_PIB','Financiero_pct_PIB',
                    'Ingresos_real','Gasto_primario_real','Intereses_real']].copy()
tabla1.columns = ['Primario (B)', 'Financiero (B)', 'Primario % PIB', 'Financiero % PIB',
                   'Ingresos (B)', 'Gasto primario (B)', 'Intereses (B)']
tabla1 = tabla1[tabla1.index.isin(range(2020,2027))]
print(tabla1.round(1).to_string())
print('  * PIB nominal aproximado | Fuentes: INDEC, Hacienda')

# ── 2. El ajuste Milei ────────────────────────────────────────────────
print()
print('=' * 80)
print('2. EL AJUSTE MILEI: VARIACION REAL 2023 -> 2024 / 2025')
print('   (Billones ARS de abril 2026 | Gasto primario Sector Publico Total)')
print('=' * 80)

gp23 = df_macro.loc[2023,'Gasto_primario_real']
gp24 = df_macro.loc[2024,'Gasto_primario_real']
gp25 = df_macro.loc[2025,'Gasto_primario_real']
rp23 = df_macro.loc[2023,'Primario_real']
rp24 = df_macro.loc[2024,'Primario_real']
rp25 = df_macro.loc[2025,'Primario_real']
rf23 = df_macro.loc[2023,'Financiero_real']
rf24 = df_macro.loc[2024,'Financiero_real']
rf25 = df_macro.loc[2025,'Financiero_real']

print(f'  Gasto primario 2023:  {gp23:.1f} B ARS')
print(f'  Gasto primario 2024:  {gp24:.1f} B ARS  ({(gp24-gp23)/gp23*100:+.1f}% real)')
print(f'  Gasto primario 2025:  {gp25:.1f} B ARS  ({(gp25-gp23)/gp23*100:+.1f}% real vs 2023)')
print(f'  Reduccion 2023->2024: {gp24-gp23:+.1f} B ARS')
print(f'  Reduccion 2023->2025: {gp25-gp23:+.1f} B ARS')
print()
print(f'  Resultado Primario  2023: {rp23:+.1f} B  | 2024: {rp24:+.1f} B  | 2025: {rp25:+.1f} B')
print(f'  Resultado Financiero 2023: {rf23:+.1f} B  | 2024: {rf24:+.1f} B  | 2025: {rf25:+.1f} B')
print(f'  Mejora resultado primario  2023->2024: {rp24-rp23:+.1f} B')
print(f'  Mejora resultado financiero 2023->2024: {rf24-rf23:+.1f} B')

# ── 3. Desglose funcional del recorte (IMIG) ──────────────────────────
print()
print('=' * 80)
print('3. DESGLOSE FUNCIONAL DEL RECORTE 2023->2025 (fuente: IMIG)')
print('   (Billones ARS de abril 2026)')
print('=' * 80)

# Rubros: (nombre_display, concepto_codigo, nivel_jerarquia)
# Nota: AUH usa codigo 'AUH' (normalizacion corregida)
rubros_tabla = [
    ('Obra publica (gastos capital)',    'Gastos_capital',              1),
    ('Subsidios economicos',             'Subsidios_economicos',        1),
    ('Transferencias a provincias',      'Transf_corrientes_provincias',1),
    ('Salarios estatales',               'Salarios',                    2),
    ('Jubilaciones contributivas',       'Jubilaciones_pensiones',      2),
    ('Universidades',                    'Transf_universidades',        1),
    ('Pensiones no contributivas',       'Pensiones_no_contributivas',  2),
    ('PAMI (INSSJP)',                    'INSSJP_PAMI',                 2),
    ('Otros programas sociales',         'Otros_prog_sociales',         2),
    ('AUH',                              'AUH',                         2),
]

total_ajuste_aif = gp23 - gp25

filas = []
for nombre, cod, nivel in rubros_tabla:
    anual = get_imig_anual(cod, nivel)
    v23 = anual.get(2023, None)
    v24 = anual.get(2024, None)
    v25 = anual.get(2025, None)
    if v23 is None or v25 is None:
        print(f'  [AVISO] Sin datos para {nombre} (codigo: {cod})')
        continue
    var25     = v25 - v23
    pct_var   = (v25 / v23 - 1) * 100 if v23 != 0 else 0
    pct_ajuste = -var25 / total_ajuste_aif * 100
    filas.append({
        'Rubro':                    nombre,
        '2023 (B)':                 round(v23, 1),
        '2024 (B)':                 round(v24, 1) if v24 is not None else None,
        '2025 (B)':                 round(v25, 1),
        'Variacion 23->25 (B)':     round(var25, 1),
        'Var % real':               f'{pct_var:+.1f}%',
        '% del ajuste total':       f'{pct_ajuste:+.1f}%',
    })

df_rubros = pd.DataFrame(filas)
print(f'  Reduccion total gasto primario AIF 2023->2025 = {total_ajuste_aif:.1f} B (referencia)')
print()
print(df_rubros.to_string(index=False))
print()
suma_rec = df_rubros[df_rubros['Variacion 23->25 (B)'] < 0]['Variacion 23->25 (B)'].sum()
suma_aum = df_rubros[df_rubros['Variacion 23->25 (B)'] > 0]['Variacion 23->25 (B)'].sum()
print(f'  Suma de recortes identificados en IMIG: {suma_rec:.1f} B')
print(f'  Suma de aumentos identificados en IMIG:  {suma_aum:+.1f} B (AUH, PAMI, Jubilaciones)')
print(f'  Neto IMIG:                               {suma_rec+suma_aum:.1f} B')
print()
print('  Nota: IMIG cubre sector publico consolidado; AIF es base caja.')

In [ ]:
# ── Celda 9: Analisis completo del ajuste fiscal Milei ────────────────
# Fuentes: AIF (Sector Publico Total) para macro | IMIG para funcional

# PIB nominal aproximado (en billones = 10^12 pesos)
# 2023 y 2024-2025: derivados de ratios publicados por Hacienda
# 2020-2022: estimados INDEC/Hacienda
PIB_B = {2020: 44.9, 2021: 72.0, 2022: 115.0,
         2023: 143.2, 2024: 586.7, 2025: 725.0}
# Fuente PIB: 2023 = 8.74B deficit / 6.1% (Hacienda)
#             2024 = 1.76B surplus / 0.3% (Hacienda)
#             2025 = 1.45B surplus / 0.2% (Hacienda)

# ── 1. Resultado fiscal anual ─────────────────────────────────────────
print('=' * 80)
print('1. RESULTADO FISCAL ANUAL - SECTOR PUBLICO TOTAL')
print('   (Billones ARS, real = precios abril 2026)')
print('=' * 80)

df_macro = pd.DataFrame()
for concepto, col in [
    ('XIV_RESULTADO_PRIMARIO',               'Primario_real'),
    ('XV_RESULTADO_FINANCIERO',              'Financiero_real'),
    ('I_INGRESOS_CORRIENTES',                'Ingresos_real'),
    ('XII_GASTOS_PRIMARIOS_DESPUES_FIGURAT', 'Gasto_primario_real'),
    ('II2_INTERESES',                        'Intereses_real'),
]:
    s = get_serie_total(concepto)
    real = pd.Series(deflactar(s), index=s.index)
    df_macro[col] = real.groupby(real.index.year).sum() / 1e6

# Nominales para % PIB
for concepto, col in [
    ('XIV_RESULTADO_PRIMARIO',  'Primario_nom'),
    ('XV_RESULTADO_FINANCIERO', 'Financiero_nom'),
]:
    s = get_serie_total(concepto)
    df_macro[col] = s.groupby(s.index.year).sum() / 1e6

df_macro['Primario_pct_PIB']    = df_macro.apply(
    lambda r: f"{r['Primario_nom']/PIB_B.get(r.name,1)*100:+.1f}%" if r.name in PIB_B else 'n/d', axis=1)
df_macro['Financiero_pct_PIB']  = df_macro.apply(
    lambda r: f"{r['Financiero_nom']/PIB_B.get(r.name,1)*100:+.1f}%" if r.name in PIB_B else 'n/d', axis=1)

tabla1 = df_macro[['Primario_real','Financiero_real','Primario_pct_PIB','Financiero_pct_PIB',
                    'Ingresos_real','Gasto_primario_real','Intereses_real']].copy()
tabla1.columns = ['Primario (B)', 'Financiero (B)', 'Primario % PIB', 'Financiero % PIB',
                   'Ingresos (B)', 'Gasto primario (B)', 'Intereses (B)']
tabla1 = tabla1[tabla1.index.isin(range(2020,2027))]
print(tabla1.round(1).to_string())
print('  * PIB nominal aproximado | Fuentes: INDEC, Hacienda')

# ── 2. El ajuste Milei: dic-2023 a la actualidad ──────────────────────
print()
print('=' * 80)
print('2. EL AJUSTE MILEI: VARIACION REAL 2023 -> 2024 / 2025')
print('   (Billones ARS de abril 2026 | Gasto primario Sector Publico Total)')
print('=' * 80)

gp23 = df_macro.loc[2023,'Gasto_primario_real']
gp24 = df_macro.loc[2024,'Gasto_primario_real']
gp25 = df_macro.loc[2025,'Gasto_primario_real']
rp23 = df_macro.loc[2023,'Primario_real']
rp24 = df_macro.loc[2024,'Primario_real']
rp25 = df_macro.loc[2025,'Primario_real']
rf23 = df_macro.loc[2023,'Financiero_real']
rf24 = df_macro.loc[2024,'Financiero_real']
rf25 = df_macro.loc[2025,'Financiero_real']

print(f'  Gasto primario 2023:  {gp23:.1f} B ARS')
print(f'  Gasto primario 2024:  {gp24:.1f} B ARS  ({(gp24-gp23)/gp23*100:+.1f}% real)')
print(f'  Gasto primario 2025:  {gp25:.1f} B ARS  ({(gp25-gp23)/gp23*100:+.1f}% real vs 2023)')
print(f'  Reduccion 2023->2024: {gp24-gp23:+.1f} B ARS')
print(f'  Reduccion 2023->2025: {gp25-gp23:+.1f} B ARS')
print()
print(f'  Resultado Primario 2023:  {rp23:+.1f} B  |  2024: {rp24:+.1f} B  |  2025: {rp25:+.1f} B')
print(f'  Resultado Financiero 2023: {rf23:+.1f} B  |  2024: {rf24:+.1f} B  |  2025: {rf25:+.1f} B')
print(f'  Mejora resultado primario  2023->2024: {rp24-rp23:+.1f} B')
print(f'  Mejora resultado financiero 2023->2024: {rf24-rf23:+.1f} B')

# ── 3. Desglose funcional del recorte (IMIG) ──────────────────────────
print()
print('=' * 80)
print('3. DESGLOSE FUNCIONAL DEL RECORTE 2023->2025 (fuente: IMIG)')
print('   (Billones ARS de abril 2026)')
print('=' * 80)

rubros_tabla = [
    ('Obra publica (gastos capital)',    'Gastos_capital',              1),
    ('Subsidios economicos',             'Subsidios_economicos',        1),
    ('Transferencias a provincias',      'Transf_corrientes_provincias',1),
    ('Salarios estatales',               'Salarios',                    2),
    ('Jubilaciones contributivas',       'Jubilaciones_pensiones',      2),
    ('Universidades',                    'Transf_universidades',        1),
    ('Pensiones no contributivas',       'Pensiones_no_contributivas',  2),
    ('PAMI (INSSJP)',                    'INSSJP_PAMI',                 2),
    ('AUH',                              'Asignación Universal para Protección Social', 2),
]

total_ajuste_aif = gp23 - gp25  # reduccion total en terminos reales

filas = []
for nombre, cod, nivel in rubros_tabla:
    anual = get_imig_anual(cod, nivel)
    v23 = anual.get(2023, None)
    v24 = anual.get(2024, None)
    v25 = anual.get(2025, None)
    if v23 is None or v25 is None:
        continue
    var25 = v25 - v23
    pct_var = (v25 / v23 - 1) * 100 if v23 != 0 else 0
    pct_ajuste = -var25 / total_ajuste_aif * 100  # contribucion al recorte
    filas.append({
        'Rubro': nombre,
        '2023 (B)': round(v23, 1),
        '2024 (B)': round(v24, 1) if v24 is not None else None,
        '2025 (B)': round(v25, 1),
        'Variacion 23->25 (B)': round(var25, 1),
        'Var % real': f'{pct_var:+.1f}%',
        '% del ajuste total': f'{pct_ajuste:+.1f}%',
    })

df_rubros = pd.DataFrame(filas)
print(f'  Referencia: reduccion total gasto primario AIF 2023->2025 = {total_ajuste_aif:.1f} B')
print()
print(df_rubros.to_string(index=False))
print()
suma_recortes = df_rubros[df_rubros['Variacion 23->25 (B)'] < 0]['Variacion 23->25 (B)'].sum()
suma_aumentos = df_rubros[df_rubros['Variacion 23->25 (B)'] > 0]['Variacion 23->25 (B)'].sum()
print(f'  Suma de recortes identificados en IMIG: {suma_recortes:.1f} B')
print(f'  Suma de aumentos identificados en IMIG:  {suma_aumentos:+.1f} B (AUH, PAMI)')
print(f'  Neto IMIG:                              {suma_recortes+suma_aumentos:.1f} B')
print()
print('  Nota: el IMIG cubre el sector publico consolidado; el AIF es base caja.')
print('  Diferencia entre totales refleja distintos perimetros y metodologias.')

In [ ]:
# ── Celda 10: Exportar Excel + ZIP ────────────────────────────────────
EXCEL_FILE = 'analisis_fiscal_resultados.xlsx'
ZIP_FILE   = 'analisis_fiscal.zip'

# Serie mensual KPIs (Sector Publico Total)
idx_base = get_serie_total('XIV_RESULTADO_PRIMARIO').index
rows_m   = {'fecha': idx_base}
for nombre, cod in [
    ('ingresos_corrientes',  'I_INGRESOS_CORRIENTES'),
    ('gastos_corrientes',    'II_GASTOS_CORRIENTES'),
    ('intereses',            'II2_INTERESES'),
    ('prestaciones',         'II3_PRESTACIONES_SEG_SOCIAL'),
    ('resultado_primario',   'XIV_RESULTADO_PRIMARIO'),
    ('resultado_financiero', 'XV_RESULTADO_FINANCIERO'),
]:
    s = get_serie_total(cod)
    rows_m[f'{nombre}_nominal_MM_ARS'] = s.reindex(idx_base).values
    rows_m[f'{nombre}_real_MM_ARS']    = deflactar(s.reindex(idx_base))
df_serie = pd.DataFrame(rows_m)

# Resumen anual
df_anual_exp = df_serie.copy()
df_anual_exp['anio'] = pd.to_datetime(df_anual_exp['fecha']).dt.year
df_anual_exp = df_anual_exp.groupby('anio').sum(numeric_only=True)

# Transferencias a provincias
prov_rows = []
for cod, tipo in [('II4b1_TRANSF_PROVINCIAS_CABA','Corrientes'),
                  ('V2a_TRANSF_CAPITAL_PROVINCIAS','Capital')]:
    for ss in ['tesoro_nacional','total_adm_nacional']:
        s = get_serie(cod, ss)
        if len(s) == 0: continue
        prov_rows.append(pd.DataFrame({
            'fecha': s.index, 'tipo': tipo, 'subsector': ss,
            'nominal_MM_ARS': s.values, 'real_MM_ARS': deflactar(s),
        }))
df_prov_exp = pd.concat(prov_rows).sort_values(['fecha','tipo'])

# Desglose funcional
df_rubros_exp = df_rubros.copy()

with pd.ExcelWriter(EXCEL_FILE, engine='openpyxl') as writer:
    df_serie.to_excel(writer,      sheet_name='Serie_mensual',       index=False)
    df_anual_exp.to_excel(writer,  sheet_name='Resumen_anual',       index=True)
    df_prov_exp.to_excel(writer,   sheet_name='Transferencias_prov', index=False)
    df_ajuste.reset_index().to_excel(writer, sheet_name='Ajuste_AIF',index=False)
    df_rubros_exp.to_excel(writer, sheet_name='Ajuste_IMIG_funcional',index=False)

print(f'Excel generado: {EXCEL_FILE}')
print('Hojas: Serie_mensual | Resumen_anual | Transferencias_prov | Ajuste_AIF | Ajuste_IMIG_funcional')

# ZIP con graficos + Excel
with zipfile.ZipFile(ZIP_FILE, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(EXCEL_FILE)
    for g in GRAFICOS:
        if os.path.exists(g):
            zf.write(g)

print(f'\nZIP generado: {ZIP_FILE}')
print(f'Contenido ({1 + len([g for g in GRAFICOS if os.path.exists(g)])} archivos):')
print(f'  {EXCEL_FILE}')
for g in GRAFICOS:
    if os.path.exists(g):
        print(f'  {g}  ({os.path.getsize(g)//1024} KB)')

if IN_COLAB:
    from google.colab import files
    files.download(ZIP_FILE)
    print('\nDescarga iniciada')
else:
    print(f'\nArchivo en: {os.path.abspath(ZIP_FILE)}')